# Notebook 03 — Imbalance Ablation (Natural vs Class-Weights vs SMOTE)
### Purpose: test whether SMOTE trades calibration for recall — justifying the calibration step

In [30]:
import os
for root, _, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv'):
            print(os.path.join(root, f))

/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/cell_train.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/maven_churn_reason.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/maven_cal.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/maven_test.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/cell_cal.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/cell_feature_meta.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/maven_train.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/maven_feature_meta.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1/cell_test.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-2/results_baseline.csv
/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-2/results_cv_auc.csv


In [32]:
# ===== CELL 1: imports + config =====
import os, warnings
import numpy as np
import pandas as pd
import joblib
warnings.filterwarnings("ignore")
 
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             recall_score, precision_score, brier_score_loss,
                             log_loss, accuracy_score)
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
 
SEED = 42
np.random.seed(SEED)
INPUT_DIR = "/kaggle/input/notebooks/mumtaheenabinteahmed/ieee-csde-1"
OUT_DIR   = "/kaggle/working"
 
DATASETS = {
    "maven": ("maven_train.csv", "maven_cal.csv", "maven_test.csv"),
    "cell":  ("cell_train.csv",  "cell_cal.csv",  "cell_test.csv"),
}
STRATEGIES = ["natural", "classweight", "smote"]
 

In [33]:
# ===== CELL 2: helpers (self-contained) =====
def load_split(name):
    tr, ca, te = DATASETS[name]
    return (pd.read_csv(f"{INPUT_DIR}/{tr}"),
            pd.read_csv(f"{INPUT_DIR}/{ca}"),
            pd.read_csv(f"{INPUT_DIR}/{te}"))
 
def split_xy(df):
    return df.drop(columns=["target"]), df["target"].values
 
def build_preprocessor(X, scale=False):
    num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
    num_steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer([
        ("num", Pipeline(num_steps), num_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
    ])
 
def ece_score(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.digitize(y_prob, bins) - 1, 0, n_bins - 1)
    ece, n = 0.0, len(y_true)
    for b in range(n_bins):
        m = idx == b
        if m.sum():
            ece += (m.sum() / n) * abs(y_true[m].mean() - y_prob[m].mean())
    return ece
 
def evaluate(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "AUC": roc_auc_score(y_true, y_prob),
        "PR_AUC": average_precision_score(y_true, y_prob),
        "F1": f1_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Brier": brier_score_loss(y_true, y_prob),
        "LogLoss": log_loss(y_true, y_prob, labels=[0, 1]),
        "ECE": ece_score(y_true, y_prob),
    }
 

In [34]:
# ===== CELL 3: model factory per imbalance strategy =====
def make_base_model(name, strategy, y_tr):
    """Return (clf, needs_scale). For 'classweight' set weights; for 'smote'/'natural'
       use unweighted models (SMOTE handles imbalance via resampling)."""
    weighted = (strategy == "classweight")
    n_neg, n_pos = int((y_tr == 0).sum()), int((y_tr == 1).sum())
    spw = n_neg / max(n_pos, 1)  # scale_pos_weight for boosting
 
    if name == "LogReg":
        clf = LogisticRegression(max_iter=2000, random_state=SEED,
                                 class_weight="balanced" if weighted else None)
        return clf, True
    if name == "RF":
        clf = RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=SEED,
                                     class_weight="balanced" if weighted else None)
        return clf, False
    if name == "XGB":
        clf = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                            subsample=0.9, colsample_bytree=0.9, tree_method="hist",
                            eval_metric="logloss", random_state=SEED, n_jobs=-1,
                            scale_pos_weight=spw if weighted else 1.0)
        return clf, False
    if name == "LGBM":
        clf = LGBMClassifier(n_estimators=400, learning_rate=0.05, subsample=0.9,
                             colsample_bytree=0.9, random_state=SEED, n_jobs=-1, verbose=-1,
                             class_weight="balanced" if weighted else None)
        return clf, False
 
MODEL_NAMES = ["LogReg", "RF", "XGB", "LGBM"]
 
def build_pipeline(X_tr, y_tr, name, strategy):
    clf, needs_scale = make_base_model(name, strategy, y_tr)
    pre = build_preprocessor(X_tr, scale=needs_scale)
    if strategy == "smote":
        # SMOTE resamples train only during fit; runs on transformed feature matrix
        return ImbPipeline([("pre", pre), ("smote", SMOTE(random_state=SEED)), ("clf", clf)])
    return Pipeline([("pre", pre), ("clf", clf)])

In [35]:

# ===== CELL 4: run ablation =====
rows = []
for ds in DATASETS:
    train, cal, test = load_split(ds)
    X_tr, y_tr = split_xy(train)
    X_ca, y_ca = split_xy(cal)
    X_te, y_te = split_xy(test)
    print(f"\n===== {ds.upper()} =====")
    for name in MODEL_NAMES:
        for strat in STRATEGIES:
            pipe = build_pipeline(X_tr, y_tr, name, strat)
            pipe.fit(X_tr, y_tr)
            p_ca = pipe.predict_proba(X_ca)[:, 1]
            p_te = pipe.predict_proba(X_te)[:, 1]
 
            m = evaluate(y_te, p_te)
            m.update({"dataset": ds, "model": name, "strategy": strat})
            rows.append(m)
 
            np.save(f"{OUT_DIR}/prob_{ds}_{name}_{strat}_cal.npy",  p_ca)
            np.save(f"{OUT_DIR}/prob_{ds}_{name}_{strat}_test.npy", p_te)
 
            print(f"  {name:6s} {strat:11s} | AUC={m['AUC']:.4f} "
                  f"Recall={m['Recall']:.4f} F1={m['F1']:.4f} "
                  f"Brier={m['Brier']:.4f} ECE={m['ECE']:.4f}")


===== MAVEN =====
  LogReg natural     | AUC=0.9043 Recall=0.6952 F1=0.7075 Brier=0.1097 ECE=0.0309
  LogReg classweight | AUC=0.9039 Recall=0.8717 F1=0.7318 Brier=0.1287 ECE=0.0999
  LogReg smote       | AUC=0.9034 Recall=0.8422 F1=0.7175 Brier=0.1267 ECE=0.0896
  RF     natural     | AUC=0.9190 Recall=0.6658 F1=0.7313 Brier=0.0988 ECE=0.0232
  RF     classweight | AUC=0.9198 Recall=0.6524 F1=0.7230 Brier=0.0992 ECE=0.0260
  RF     smote       | AUC=0.9164 Recall=0.6952 F1=0.7471 Brier=0.1004 ECE=0.0354
  XGB    natural     | AUC=0.9251 Recall=0.7139 F1=0.7596 Brier=0.0955 ECE=0.0346
  XGB    classweight | AUC=0.9235 Recall=0.7834 F1=0.7571 Brier=0.1017 ECE=0.0535
  XGB    smote       | AUC=0.9244 Recall=0.7246 F1=0.7612 Brier=0.0948 ECE=0.0381
  LGBM   natural     | AUC=0.9252 Recall=0.7273 F1=0.7566 Brier=0.0986 ECE=0.0575
  LGBM   classweight | AUC=0.9228 Recall=0.7674 F1=0.7583 Brier=0.1027 ECE=0.0641
  LGBM   smote       | AUC=0.9222 Recall=0.7219 F1=0.7521 Brier=0.1012 ECE=0.05

In [36]:
# ===== CELL 5: full results table =====
res = pd.DataFrame(rows)[
    ["dataset", "model", "strategy", "AUC", "PR_AUC", "F1", "Recall",
     "Precision", "Brier", "LogLoss", "ECE"]
].round(4)
res.to_csv(f"{OUT_DIR}/results_imbalance.csv", index=False)
print("\n============== IMBALANCE ABLATION (test) ==============")
print(res.to_string(index=False))


============== IMBALANCE ABLATION (test) ==============
dataset  model    strategy    AUC  PR_AUC     F1  Recall  Precision  Brier  LogLoss    ECE
  maven LogReg     natural 0.9043  0.7961 0.7075  0.6952     0.7202 0.1097   0.3490 0.0309
  maven LogReg classweight 0.9039  0.7932 0.7318  0.8717     0.6306 0.1287   0.3937 0.0999
  maven LogReg       smote 0.9034  0.7884 0.7175  0.8422     0.6250 0.1267   0.3894 0.0896
  maven     RF     natural 0.9190  0.8471 0.7313  0.6658     0.8111 0.0988   0.3148 0.0232
  maven     RF classweight 0.9198  0.8455 0.7230  0.6524     0.8106 0.0992   0.3154 0.0260
  maven     RF       smote 0.9164  0.8427 0.7471  0.6952     0.8075 0.1004   0.3225 0.0354
  maven    XGB     natural 0.9251  0.8621 0.7596  0.7139     0.8116 0.0955   0.3085 0.0346
  maven    XGB classweight 0.9235  0.8587 0.7571  0.7834     0.7325 0.1017   0.3239 0.0535
  maven    XGB       smote 0.9244  0.8637 0.7612  0.7246     0.8018 0.0948   0.3092 0.0381
  maven   LGBM     natural 0.9252

In [37]:
# ===== CELL 6: the money view — does SMOTE trade calibration for recall? =====
# Averaged across models, per dataset x strategy: Recall (up?) vs Brier/ECE (worse?)
summary = (res.groupby(["dataset", "strategy"])[["Recall", "F1", "Brier", "ECE", "AUC"]]
              .mean().round(4).reset_index())
# order strategies naturally
order = {"natural": 0, "classweight": 1, "smote": 2}
summary["_o"] = summary["strategy"].map(order)
summary = summary.sort_values(["dataset", "_o"]).drop(columns="_o").reset_index(drop=True)
summary.to_csv(f"{OUT_DIR}/results_imbalance_summary.csv", index=False)
print("\n===== TRADEOFF SUMMARY (mean across models) =====")
print(summary.to_string(index=False))
print("\nRead: if SMOTE rows show higher Recall but higher Brier/ECE (worse "
      "calibration) vs natural/classweight, the calibration step is justified.")
 
# ============================================================================
# DONE. Next: Notebook 04 — post-hoc calibration (Platt/isotonic) fit on CAL
# split, applied to each (model x strategy). Show calibration restores Brier/ECE,
# especially for the SMOTE variants. Then Notebook 05 ties it to profit.
# ============================================================================


===== TRADEOFF SUMMARY (mean across models) =====
dataset    strategy  Recall     F1  Brier    ECE    AUC
   cell     natural  0.1036 0.1706 0.1902 0.0130 0.6656
   cell classweight  0.4550 0.3805 0.2147 0.1366 0.6652
   cell       smote  0.2452 0.2702 0.2009 0.0610 0.6622
  maven     natural  0.7006 0.7388 0.1006 0.0366 0.9184
  maven classweight  0.7687 0.7426 0.1081 0.0609 0.9175
  maven       smote  0.7460 0.7445 0.1058 0.0550 0.9166

Read: if SMOTE rows show higher Recall but higher Brier/ECE (worse calibration) vs natural/classweight, the calibration step is justified.
